# ⚡ FlashAttention Forward Pass in Triton — Branching vs. Staged Kernels

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AvinashQtC/flash-attention-triton/blob/main/flash_attention_forward_triton.ipynb)

A from-scratch **causal FlashAttention forward kernel** written in [Triton](https://triton-lang.org),
in two versions that compute the same thing but organise the causal mask differently:

| | Kernel | How the causal mask is handled |
|---|---|---|
| **V1** | `_fwd_v1_branch` | One loop over all KV tiles; a **runtime `if`** inside the loop masks tiles that touch the diagonal. Every tile pays for boundary masks. |
| **V2** | `_fwd_v2_staged` | The loop is split into **two compile-time stages** (`STAGE: tl.constexpr`): off-diagonal tiles run with **no masks at all**, only the diagonal tile is masked. No branch in the hot loop. |

Both kernels are validated against an explicit FP64 attention reference. The tests distinguish FP16,
strict FP32 IEEE, and FP32 storage with TF32 tensor-core dot products. Benchmarks compare both Triton
kernels with PyTorch's automatic SDPA dispatcher. The current target is an **NVIDIA L40S** (Ada, `sm_89`).

**Contents**
1. [Setup](#1-setup)
2. [The algorithm: tiling + online softmax](#2-the-algorithm-tiling--online-softmax)
3. [A NumPy reference of the tiled algorithm](#3-a-numpy-reference-of-the-tiled-algorithm)
4. [Triton kernels](#4-triton-kernels)
5. [Correctness tests](#5-correctness-tests)
6. [Benchmark](#6-benchmark)
7. [Discussion & next steps](#7-discussion--next-steps)

> **Run it:** select a CUDA GPU. The `fp32_tf32` mode requires a TF32-capable NVIDIA GPU (Ampere or newer).

## 1. Setup

In [1]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

name, memory.total [MiB], driver_version
NVIDIA L40S, 46068 MiB, 595.91.07
NVIDIA L40S, 46068 MiB, 595.91.07


In [2]:
import math, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import triton
import triton.language as tl
import triton.testing

# Strict FP32 by default. The benchmark (Section 6) switches PyTorch to TF32 only for the TF32 suite.
# Use the new fp32_precision API only; mixing it with the legacy allow_tf32 flags is an error in recent PyTorch.
torch.backends.cuda.matmul.fp32_precision = "ieee"
assert torch.cuda.is_available(), "A CUDA GPU is required."
DEVICE = torch.device("cuda")
props = torch.cuda.get_device_properties(DEVICE)
print(f"GPU     : {props.name}  (sm_{props.major}{props.minor}, {props.multi_processor_count} SMs, "
      f"{props.total_memory / 2**30:.1f} GiB)")
print(f"torch   : {torch.__version__}")
print(f"triton  : {triton.__version__}")
torch.manual_seed(0)

GPU     : NVIDIA L40S  (sm_89, 142 SMs, 44.4 GiB)
torch   : 2.13.0+cu130
triton  : 3.7.1


## 2. The algorithm: tiling + online softmax

For one (batch, head) with $Q, K, V \in \mathbb{R}^{N \times d}$, causal attention is

$$
S = \frac{QK^\top}{\sqrt{d}}, \qquad
S_{ij} \leftarrow -\infty \ \text{ if } j > i, \qquad
P = \operatorname{softmax}_{\text{row}}(S), \qquad
O = PV .
$$

The naive implementation materialises $S$ and $P$ — an $N \times N$ matrix per head — in GPU memory (HBM).
FlashAttention never does: it streams $K, V$ through on-chip SRAM in tiles and keeps a **running softmax**.

### Online softmax

Split the keys into tiles $K_1, K_2, \dots$ of width $B_{kv}$. For a block of $B_q$ query rows keep three
running quantities per row: the max $m$, the normaliser $\ell$, and the un-normalised output $\mathrm{acc}$.
For tile $j$ with scores $S_j = Q K_j^\top / \sqrt d$:

$$
\begin{aligned}
m^{\text{new}} &= \max\!\big(m,\ \operatorname{rowmax}(S_j)\big) \\
\alpha &= e^{\,m - m^{\text{new}}} &&\text{(rescales everything seen so far)}\\
\tilde P_j &= e^{\,S_j - m^{\text{new}}} \\
\ell &\leftarrow \alpha\,\ell + \operatorname{rowsum}(\tilde P_j) \\
\mathrm{acc} &\leftarrow \alpha\,\mathrm{acc} + \tilde P_j V_j \\
m &\leftarrow m^{\text{new}}
\end{aligned}
$$

The invariant after tile $j$ is $\ \mathrm{acc} = \sum_{k \le j} e^{S_k - m} V_k,\ \ \ell = \sum_{k \le j} \operatorname{rowsum}\, e^{S_k - m}$,
so at the end

$$
O = \mathrm{acc} / \ell, \qquad L = m + \log \ell = \operatorname{logsumexp}_{\text{row}}(S).
$$

$L$ is stored because the backward pass can rebuild $P = e^{S - L}$ from it without another reduction.

### Softmax in base 2

GPUs evaluate $e^x$ as $2^{x \log_2 e}$ on the special-function unit (`MUFU.EX2`). The kernels fold the
softmax scale and $\log_2 e$ into one constant $c = \log_2 e / \sqrt d$ and track the running max in base-2
units, $m = \max_j c\,(QK^\top)_{ij}$:

$$
\tilde P_j = 2^{\,c\,(QK_j^\top) - m^{\text{new}}}, \qquad
\alpha = 2^{\,m - m^{\text{new}}}, \qquad
L = \ln 2 \,\big(m + \log_2 \ell\big).
$$

`qk * c - m_new` compiles to a single FMA, so each score costs one FFMA plus one `MUFU.EX2`, instead of a
multiply, a subtract and a second multiply. The returned $L$ is still the natural-log logsumexp.

### Where the causal mask matters

With $B_q = B_{kv}$, query tile $i$ only needs key tiles $j \le i$:

```
            key tiles  j →
            0   1   2   3
query  0  [ D ] ·   ·   ·        F = fully visible (no mask needed)
tiles  1  [ F ][ D ] ·   ·       D = diagonal tile (causal mask needed)
  i    2  [ F ][ F ][ D ] ·      · = skipped entirely
  ↓    3  [ F ][ F ][ F ][ D ]
```

Only the $n$ diagonal tiles out of $n(n+1)/2$ computed tiles need masking — a fraction $2/(n+1)$ that shrinks
as the sequence grows. **V2 exploits this**; V1 checks every tile at runtime.

Number of computed tiles vs. tiles that actually need a mask (for $B_q = B_{kv} = 64$):

In [3]:
BLOCK = 64
rows = []
for N in [512, 1024, 2048, 4096, 8192]:
    n = math.ceil(N / BLOCK)
    computed = n * (n + 1) // 2
    rows.append({"SEQ_LEN": N, "tiles computed": computed, "diagonal (masked)": n,
                 "masked fraction": f"{n / computed:.1%}"})
pd.DataFrame(rows).set_index("SEQ_LEN")

,tiles computed,diagonal (masked),masked fraction
SEQ_LEN,,,
512,36,8,22.2%
1024,136,16,11.8%
2048,528,32,6.1%
4096,2080,64,3.1%
8192,8256,128,1.6%


## 3. A NumPy reference of the tiled algorithm

Before writing a GPU kernel, the same loop structure in plain NumPy — one query tile at a time, the two
stages of V2 made explicit. It runs on CPU and is checked against the naive $N \times N$ formula.

In [4]:
def attention_naive_np(q, k, v):
    """q, k, v: [N, d]. Materialises the full N x N score matrix."""
    N, d = q.shape
    s = q @ k.T / np.sqrt(d)
    s[np.triu_indices(N, k=1)] = -np.inf
    smax = s.max(axis=1, keepdims=True)
    p = np.exp(s - smax)
    lse = smax[:, 0] + np.log(p.sum(axis=1))
    return (p / p.sum(axis=1, keepdims=True)) @ v, lse


def attention_tiled_np(q, k, v, block_q=64, block_kv=32):
    """Causal FlashAttention forward, written exactly like the V2 kernel (base-2 softmax)."""
    N, d = q.shape
    assert block_q % block_kv == 0
    qk_scale = np.log2(np.e) / np.sqrt(d)                    # softmax scale and log2(e) folded together
    out, lse = np.empty_like(q), np.empty(N)

    for q_start in range(0, N, block_q):                     # one Triton program per query tile
        rows = np.arange(q_start, min(q_start + block_q, N))
        qi = q[rows]
        m = np.full(len(rows), -np.inf)
        l = np.zeros(len(rows))
        acc = np.zeros((len(rows), d))

        for stage in (1, 2):
            lo, hi = (0, q_start) if stage == 1 else (q_start, q_start + block_q)
            for kv_start in range(lo, min(hi, N), block_kv):
                cols = np.arange(kv_start, min(kv_start + block_kv, N))
                s = qi @ k[cols].T                             # raw scores; scaling happens in the FMA below
                if stage == 2:                                 # diagonal tile only
                    s = np.where(cols[None, :] <= rows[:, None], s, -np.inf)
                m_new = np.maximum(m, s.max(axis=1) * qk_scale)  # running max in base-2 units
                p = np.exp2(s * qk_scale - m_new[:, None])      # one FMA + one exp2 per score
                alpha = np.exp2(m - m_new)
                l = alpha * l + p.sum(axis=1)
                acc = alpha[:, None] * acc + p @ v[cols]
                m = m_new

        out[rows] = acc / l[:, None]
        lse[rows] = np.log(2.0) * (m + np.log2(l))           # back to natural-log LSE
    return out, lse


rng = np.random.default_rng(0)
for N in [64, 200, 333]:
    q, k, v = (rng.standard_normal((N, 32)) for _ in range(3))
    o_tiled, lse_tiled = attention_tiled_np(q, k, v)
    o_ref, lse_ref = attention_naive_np(q, k, v)
    print(f"N={N:4d}  max|ΔO| = {np.abs(o_tiled - o_ref).max():.2e}   "
          f"max|ΔL| = {np.abs(lse_tiled - lse_ref).max():.2e}")

N=  64  max|ΔO| = 5.00e-16   max|ΔL| = 8.88e-16
N= 200  max|ΔO| = 6.66e-16   max|ΔL| = 8.88e-16
N= 333  max|ΔO| = 5.55e-16   max|ΔL| = 1.78e-15


## 4. Triton kernels

**Launch grid:** `(cdiv(SEQ_LEN, BLOCK_Q), BATCH * NUM_HEAD)` — each program owns one query tile of one head.

**Layout:** tensors are `[BATCH, NUM_HEAD, SEQ_LEN, HEAD_DIM]`. Pointer offsets use all four strides, so
non-contiguous inputs (e.g. a `[B, S, H, D]` tensor viewed with `.transpose(1, 2)`) work without a copy.

### 4.1 Autotuning space

In [5]:
# Tile configurations tried by the autotuner.
# BLOCK_Q % BLOCK_KV == 0 keeps the V2 diagonal stage on a KV-tile boundary.
# Triton discards configurations that exceed the target GPU's resource limits.
AUTOTUNE_CONFIGS = [
    triton.Config({"BLOCK_Q": 64,  "BLOCK_KV": 32},  num_warps=4, num_stages=2),
    triton.Config({"BLOCK_Q": 64,  "BLOCK_KV": 64},  num_warps=4, num_stages=2),
    triton.Config({"BLOCK_Q": 128, "BLOCK_KV": 64},  num_warps=4, num_stages=2),
    triton.Config({"BLOCK_Q": 128, "BLOCK_KV": 64},  num_warps=8, num_stages=3),
    triton.Config({"BLOCK_Q": 128, "BLOCK_KV": 128}, num_warps=8, num_stages=3),
]
AUTOTUNE_KEY = ["SEQ_LEN", "HEAD_DIM"]

### 4.2 V1 — single loop with a runtime branch

The loop runs over every KV tile up to the end of the query tile. Inside it, a **data-dependent `if`** decides
whether the tile crosses the diagonal. Because `start_kv` and `q_start` are runtime values, the compiler keeps
the branch in the loop body, and every tile still does boundary-masked loads and a `tl.where` on the scores.

In [6]:
@triton.autotune(configs=AUTOTUNE_CONFIGS, key=AUTOTUNE_KEY)
@triton.jit
def _fwd_v1_branch(
    Q_ptr, K_ptr, V_ptr, O_ptr, L_ptr,
    qk_scale,  # log2(e) / sqrt(HEAD_DIM)
    NUM_HEAD, SEQ_LEN,
    stride_b, stride_h, stride_s, stride_d,
    HEAD_DIM: tl.constexpr, BLOCK_Q: tl.constexpr, BLOCK_KV: tl.constexpr,
    USE_TF32: tl.constexpr,
    LN2: tl.constexpr = 0.6931471805599453,
):
    """V1: one KV loop with a runtime branch at the causal diagonal."""
    pid_q = tl.program_id(0)
    pid_bh = tl.program_id(1)
    b, h = pid_bh // NUM_HEAD, pid_bh % NUM_HEAD
    base = b * stride_b + h * stride_h

    offs_q = pid_q * BLOCK_Q + tl.arange(0, BLOCK_Q)
    offs_kv = tl.arange(0, BLOCK_KV)
    offs_d = tl.arange(0, HEAD_DIM)

    q_mask = offs_q[:, None] < SEQ_LEN
    q = tl.load(
        Q_ptr + base + offs_q[:, None] * stride_s + offs_d[None, :] * stride_d,
        mask=q_mask,
        other=0.0,
    )

    m = tl.full((BLOCK_Q,), float("-inf"), dtype=tl.float32)
    l = tl.zeros((BLOCK_Q,), dtype=tl.float32)
    acc = tl.zeros((BLOCK_Q, HEAD_DIM), dtype=tl.float32)

    q_start = pid_q * BLOCK_Q
    q_end = q_start + BLOCK_Q
    for start_kv in range(0, q_end, BLOCK_KV):
        kv_pos = start_kv + offs_kv
        kT = tl.load(
            K_ptr + base + kv_pos[None, :] * stride_s + offs_d[:, None] * stride_d,
            mask=kv_pos[None, :] < SEQ_LEN,
            other=0.0,
        )
        v = tl.load(
            V_ptr + base + kv_pos[:, None] * stride_s + offs_d[None, :] * stride_d,
            mask=kv_pos[:, None] < SEQ_LEN,
            other=0.0,
        )

        if USE_TF32:
            s = tl.dot(q, kT, input_precision="tf32")
        else:
            s = tl.dot(q, kT, input_precision="ieee")

        if start_kv + BLOCK_KV - 1 > q_start:
            s = tl.where(kv_pos[None, :] > offs_q[:, None], -1.0e6, s)
        s = tl.where(kv_pos[None, :] < SEQ_LEN, s, -1.0e6)

        # Scale and log2(e) are folded into qk_scale: one FMA + one MUFU.EX2 per score.
        m_new = tl.maximum(m, tl.max(s, axis=1) * qk_scale)
        p = tl.exp2(s * qk_scale - m_new[:, None])
        alpha = tl.exp2(m - m_new)
        l = alpha * l + tl.sum(p, axis=1)
        if USE_TF32:
            pv = tl.dot(p.to(v.dtype), v, input_precision="tf32")
        else:
            pv = tl.dot(p.to(v.dtype), v, input_precision="ieee")
        acc = acc * alpha[:, None] + pv
        m = m_new

    acc = acc / l[:, None]
    tl.store(L_ptr + pid_bh * SEQ_LEN + offs_q, (m + tl.log2(l)) * LN2, mask=offs_q < SEQ_LEN)
    tl.store(
        O_ptr + base + offs_q[:, None] * stride_s + offs_d[None, :] * stride_d,
        acc.to(O_ptr.dtype.element_ty),
        mask=q_mask,
    )

### 4.3 V2 — two compile-time stages, no branch in the hot loop

The KV sweep moves into a helper `_attn_inner(..., STAGE)`. `STAGE` is a `tl.constexpr`, so Triton compiles
**two specialised loops**:

* **Stage 1** — tiles `[0, q_start)`. Every key is `< q_start < SEQ_LEN`, so the tile is in-bounds and fully
  visible: plain unmasked loads, no `tl.where`.
* **Stage 2** — the single diagonal tile `[q_start, q_start + BLOCK_Q)`: one combined causal + boundary mask.

The running state `(acc, l, m)` is passed in and returned, and normalisation `acc / l` happens **once**, in the
caller, after both stages.

Both kernels use the base-2 softmax from Section 2: the launcher passes `qk_scale = log2(e)/sqrt(d)`,
the running max `m` lives in base-2 units, and the stored LSE is converted back with `ln 2 · (m + log2 ℓ)`.

In [7]:
@triton.jit
def _attn_inner(
    acc, l, m, q,
    K_ptr, V_ptr, base, offs_q, offs_kv, offs_d,
    q_start, qk_scale, SEQ_LEN, stride_s, stride_d,
    BLOCK_Q: tl.constexpr, BLOCK_KV: tl.constexpr, STAGE: tl.constexpr,
    USE_TF32: tl.constexpr,
):
    """Process off-diagonal (stage 1) or diagonal (stage 2) KV tiles."""
    if STAGE == 1:
        lo, hi = 0, q_start
    else:
        lo, hi = q_start, q_start + BLOCK_Q

    for start_kv in range(lo, hi, BLOCK_KV):
        kv_pos = start_kv + offs_kv
        k_ptrs = K_ptr + base + kv_pos[None, :] * stride_s + offs_d[:, None] * stride_d
        v_ptrs = V_ptr + base + kv_pos[:, None] * stride_s + offs_d[None, :] * stride_d

        if STAGE == 1:
            kT = tl.load(k_ptrs)
            v = tl.load(v_ptrs)
        else:
            kT = tl.load(k_ptrs, mask=kv_pos[None, :] < SEQ_LEN, other=0.0)
            v = tl.load(v_ptrs, mask=kv_pos[:, None] < SEQ_LEN, other=0.0)

        if USE_TF32:
            s = tl.dot(q, kT, input_precision="tf32")
        else:
            s = tl.dot(q, kT, input_precision="ieee")

        if STAGE == 2:
            visible = (kv_pos[None, :] <= offs_q[:, None]) & (kv_pos[None, :] < SEQ_LEN)
            s = tl.where(visible, s, -1.0e6)

        # Scale and log2(e) are folded into qk_scale: one FMA + one MUFU.EX2 per score.
        m_new = tl.maximum(m, tl.max(s, axis=1) * qk_scale)
        p = tl.exp2(s * qk_scale - m_new[:, None])
        alpha = tl.exp2(m - m_new)
        l = alpha * l + tl.sum(p, axis=1)
        if USE_TF32:
            pv = tl.dot(p.to(v.dtype), v, input_precision="tf32")
        else:
            pv = tl.dot(p.to(v.dtype), v, input_precision="ieee")
        acc = acc * alpha[:, None] + pv
        m = m_new

    return acc, l, m


@triton.autotune(configs=AUTOTUNE_CONFIGS, key=AUTOTUNE_KEY)
@triton.jit
def _fwd_v2_staged(
    Q_ptr, K_ptr, V_ptr, O_ptr, L_ptr,
    qk_scale,  # log2(e) / sqrt(HEAD_DIM)
    NUM_HEAD, SEQ_LEN,
    stride_b, stride_h, stride_s, stride_d,
    HEAD_DIM: tl.constexpr, BLOCK_Q: tl.constexpr, BLOCK_KV: tl.constexpr,
    USE_TF32: tl.constexpr,
    LN2: tl.constexpr = 0.6931471805599453,
):
    """V2: split the KV sweep into branch-free off-diagonal and diagonal stages."""
    pid_q = tl.program_id(0)
    pid_bh = tl.program_id(1)
    b, h = pid_bh // NUM_HEAD, pid_bh % NUM_HEAD
    base = b * stride_b + h * stride_h

    offs_q = pid_q * BLOCK_Q + tl.arange(0, BLOCK_Q)
    offs_kv = tl.arange(0, BLOCK_KV)
    offs_d = tl.arange(0, HEAD_DIM)

    q_mask = offs_q[:, None] < SEQ_LEN
    q = tl.load(
        Q_ptr + base + offs_q[:, None] * stride_s + offs_d[None, :] * stride_d,
        mask=q_mask,
        other=0.0,
    )

    m = tl.full((BLOCK_Q,), float("-inf"), dtype=tl.float32)
    l = tl.zeros((BLOCK_Q,), dtype=tl.float32)
    acc = tl.zeros((BLOCK_Q, HEAD_DIM), dtype=tl.float32)
    q_start = pid_q * BLOCK_Q

    acc, l, m = _attn_inner(
        acc, l, m, q, K_ptr, V_ptr, base, offs_q, offs_kv, offs_d,
        q_start, qk_scale, SEQ_LEN, stride_s, stride_d,
        BLOCK_Q, BLOCK_KV, 1, USE_TF32,
    )
    acc, l, m = _attn_inner(
        acc, l, m, q, K_ptr, V_ptr, base, offs_q, offs_kv, offs_d,
        q_start, qk_scale, SEQ_LEN, stride_s, stride_d,
        BLOCK_Q, BLOCK_KV, 2, USE_TF32,
    )

    acc = acc / l[:, None]
    tl.store(L_ptr + pid_bh * SEQ_LEN + offs_q, (m + tl.log2(l)) * LN2, mask=offs_q < SEQ_LEN)
    tl.store(
        O_ptr + base + offs_q[:, None] * stride_s + offs_d[None, :] * stride_d,
        acc.to(O_ptr.dtype.element_ty),
        mask=q_mask,
    )

### 4.4 Python launcher

In [8]:
KERNELS = {"v1_branch": _fwd_v1_branch, "v2_staged": _fwd_v2_staged}
LOG2_E = 1.4426950408889634


def flash_attention_fwd(
    q,
    k,
    v,
    version="v2_staged",
    return_lse=False,
    input_precision="ieee",
):
    """Run causal FlashAttention with explicit FP32 dot-product precision.

    ``input_precision`` must be ``"ieee"`` or ``"tf32"``. It controls both
    QK^T and P@V when inputs are float32. FP16/BF16 behavior is unchanged.
    The kernels receive qk_scale = log2(e) / sqrt(D) and run the softmax in base 2;
    the returned LSE is the natural-log logsumexp of Q K^T / sqrt(D).
    """
    assert q.shape == k.shape == v.shape, "self-attention only: q, k, v must share a shape"
    assert q.dtype == k.dtype == v.dtype, "q, k, v must have the same dtype"
    assert q.device == k.device == v.device, "q, k, v must be on the same device"
    if input_precision not in {"ieee", "tf32"}:
        raise ValueError("input_precision must be 'ieee' or 'tf32'")
    if version not in KERNELS:
        raise ValueError(f"unknown kernel version: {version}")

    B, H, S, D = q.shape
    assert D in (16, 32, 64, 128), "HEAD_DIM must be a power of two in [16, 128]"
    if not (q.stride() == k.stride() == v.stride()):
        q, k, v = q.contiguous(), k.contiguous(), v.contiguous()

    o = torch.empty_like(q)
    lse = torch.empty((B, H, S), device=q.device, dtype=torch.float32)
    grid = lambda meta: (triton.cdiv(S, meta["BLOCK_Q"]), B * H)

    KERNELS[version][grid](
        q, k, v, o, lse,
        LOG2_E / D ** 0.5,  # qk_scale
        H, S,
        *q.stride(),
        HEAD_DIM=D,
        USE_TF32=(q.dtype == torch.float32 and input_precision == "tf32"),
    )
    return (o, lse) if return_lse else o

Quick smoke test:

#### Kernel error vs. an FP64 reference

Q, K and V are upcast **once** to FP64. Every FP16 value is exactly representable in FP64, so the kernel and
the reference see the same quantized inputs and any difference comes from the kernel's own arithmetic
(FP16 matmul inputs, accumulation, online softmax, output rounding). The FP64 reference error
($\sim\sqrt N\,2^{-53}$) is ~12 orders of magnitude below an FP16 kernel's, and FP64 cannot be silently
downgraded to TF32, so no backend flags are needed here.

Two baselines bracket a correct kernel:

- `ref→fp16`: the exact output rounded to FP16, the floor no kernel can beat ($\approx |O|\cdot 2^{-12}$).
- `sdpa_fp16`: PyTorch's own FP16 path, the practical target.

A kernel 5–10× worse than `sdpa_fp16` usually accumulates $PV$ or $(m, \ell)$ in FP16; $\max|\Delta| \gtrsim 10^{-1}$
points to a logic bug (diagonal mask, missing $\alpha$ rescale, tile bounds). Reference memory grows as $N^2$
(about 4.3 GB of FP64 scores at N = 8192); for long sequences compute it per head or per query block.

In [ ]:
q = torch.randn(2, 4, 256, 64, device=DEVICE, dtype=torch.float16)
k, v = torch.randn_like(q), torch.randn_like(q)


def ref_attention_fp64(q, k, v, causal=True):
    """Exact (to FP64) causal attention. Top-left aligned mask, as in SDPA's is_causal."""
    q, k, v = q.double(), k.double(), v.double()          # exact upcast of the quantized inputs
    s = (q @ k.transpose(-2, -1)) * q.shape[-1] ** -0.5
    if causal:
        Nq, Nk = s.shape[-2:]
        mask = torch.ones(Nq, Nk, dtype=torch.bool, device=q.device).tril()  # tril(diagonal=Nk-Nq) for bottom-right
        s = s.masked_fill(~mask, float("-inf"))
    return torch.softmax(s, dim=-1) @ v


def report(name, out, ref):
    d = out.double() - ref                                 # difference computed in FP64
    print(f"{name:12s} max|Δ| = {d.abs().max().item():.2e}   "
          f"mean|Δ| = {d.abs().mean().item():.2e}   "
          f"rel L2 = {(d.norm() / ref.norm()).item():.2e}")


ref = ref_attention_fp64(q, k, v)

report("ref→fp16", ref.half(), ref)                                              # output-rounding floor
report("sdpa_fp16", F.scaled_dot_product_attention(q, k, v, is_causal=True), ref)  # practical target
for ver in KERNELS:
    report(ver, flash_attention_fwd(q, k, v, version=ver), ref)


## 5. Correctness tests

The reference explicitly computes the masked score matrix, softmax, output, and natural-log LSE in FP64.
The Triton kernels are tested as three separately labelled numerical modes:

- `fp32_ieee`: FP32 tensors with `input_precision="ieee"` for both matrix products.
- `fp32_tf32`: FP32 tensors with `input_precision="tf32"` for both matrix products.
- `fp16`: FP16 tensors; the FP32 IEEE/TF32 selector does not alter FP16 tensor-core inputs.

Each result requires finite output and LSE tensors and uses mode-specific absolute/relative tolerances.
The shapes include sequence lengths not divisible by every block size and a non-contiguous BSHD layout.

In [ ]:
def reference_attention_fp64(q, k, v):
    """Explicit causal FP64 reference for output and natural-log LSE."""
    S, D = q.shape[-2:]
    scores = (q.double() @ k.double().transpose(-1, -2)) / math.sqrt(D)
    causal_mask = torch.ones((S, S), device=q.device, dtype=torch.bool).triu(1)
    scores = scores.masked_fill(causal_mask, float("-inf"))
    probabilities = torch.softmax(scores, dim=-1)
    return probabilities @ v.double(), torch.logsumexp(scores, dim=-1)


PRECISION_MODES = {
    "fp32_ieee": {"dtype": torch.float32, "input_precision": "ieee", "atol": 2e-5, "rtol": 2e-5},
    "fp32_tf32": {"dtype": torch.float32, "input_precision": "tf32", "atol": 5e-3, "rtol": 5e-3},
    "fp16": {"dtype": torch.float16, "input_precision": "ieee", "atol": 1e-2, "rtol": 1e-2},
}


def check(B, H, S, D, mode, layout="BHSD"):
    settings = PRECISION_MODES[mode]
    dtype = settings["dtype"]
    if layout == "BHSD":
        q, k, v = (torch.randn(B, H, S, D, device=DEVICE, dtype=dtype) for _ in range(3))
    else:
        q, k, v = (
            torch.randn(B, S, H, D, device=DEVICE, dtype=dtype).transpose(1, 2)
            for _ in range(3)
        )

    ref_o, ref_lse = reference_attention_fp64(q, k, v)
    rows = []
    for version in KERNELS:
        output, lse = flash_attention_fwd(
            q,
            k,
            v,
            version=version,
            return_lse=True,
            input_precision=settings["input_precision"],
        )
        output64 = output.double()
        lse64 = lse.double()
        finite = bool(torch.isfinite(output64).all() and torch.isfinite(lse64).all())
        output_ok = torch.allclose(output64, ref_o, atol=settings["atol"], rtol=settings["rtol"])
        lse_ok = torch.allclose(lse64, ref_lse, atol=settings["atol"], rtol=settings["rtol"])
        rows.append({
            "version": version,
            "mode": mode,
            "B": B,
            "H": H,
            "S": S,
            "D": D,
            "layout": layout,
            "max|ΔO|": (output64 - ref_o).abs().max().item(),
            "max|ΔL|": (lse64 - ref_lse).abs().max().item(),
            "finite": finite,
            "pass": finite and output_ok and lse_ok,
        })
    return rows


CASES = [
    (2, 4, 256, 64),
    (1, 8, 512, 64),
    (2, 4, 300, 64),
    (1, 2, 77, 128),
    (2, 2, 1024, 16),
]
results = [row for mode in PRECISION_MODES for case in CASES for row in check(*case, mode)]
results += check(2, 4, 384, 64, "fp16", layout="BSHD")
results += check(2, 4, 384, 64, "fp32_ieee", layout="BSHD")
results += check(2, 4, 384, 64, "fp32_tf32", layout="BSHD")

tests = pd.DataFrame(results)
display(tests.style.format({"max|ΔO|": "{:.2e}", "max|ΔL|": "{:.2e}"}))
assert tests["pass"].all(), "some correctness checks failed"
print(f"All {len(tests)} IEEE/TF32/FP16 checks passed against the FP64 reference.")

## 6. Benchmark

Forward pass with `BATCH=4`, `NUM_HEAD=8`, `HEAD_DIM=64`, and causal masking. Three suites make the
storage dtype and Triton dot-product precision explicit: FP16, FP32 IEEE, and FP32 TF32. Timings use
`triton.testing.do_bench`; throughput counts useful causal FLOPs:

$$
\text{FLOPs} = \underbrace{2}_{QK^\top,\ PV} \times 2 B H N^2 d \times \underbrace{\tfrac12}_{\text{causal}} = 2BHN^2 d .
$$

**What the FLOP count leaves out.** Only the two matmuls are counted (the FlashAttention-paper convention).
The softmax work (max, FMA, `exp2`, sum, rescale) is $O(N^2)$ rather than $O(N^2 d)$, about $1/d$ of the
matmul FLOPs, but `exp2` runs on the SFU (16 results/SM/clock), which is far slower than the tensor cores.
Per score the kernel does $4d$ tensor-core FLOPs and one `exp2`, so the `exp2` time alone is roughly
$R_{\text{TC}} / (4d\,R_{\text{SFU}})$ of the matmul time. Reported TFLOP/s therefore plateaus below the
tensor-core peak even for a perfect kernel.

**Time and TFLOP/s are the same measurement.** The work is fixed by the shape, so
$\text{TFLOP/s} = \text{FLOPs}\cdot 10^{-12} / (t_{\text{ms}}\cdot 10^{-3})$ and $t = \text{FLOPs}/\text{rate}$.
For example, FP16 at $N = 8192$ is $2.749\times10^{11}$ FLOPs, and 186 TFLOP/s corresponds to about 1.48 ms.
The latency panel below is derived from the TFLOP/s table with this identity, so it carries no independent
information. TFLOP/s is the headline because time grows as $N^2$ even at constant efficiency, whereas
TFLOP/s is comparable across lengths and against the GPU's peak:
$\text{utilization} = \text{achieved TFLOP/s} / \text{peak dense tensor-core TFLOP/s}$.

The PyTorch line is labelled **PyTorch SDPA (auto)** because `scaled_dot_product_attention` dispatches to an
available backend; this notebook does not force a particular SDPA kernel. PyTorch's FP32 matmul precision is set to
match each suite (`"ieee"` for FP32 IEEE, `"tf32"` for FP32 TF32), while Triton's IEEE/TF32 choice is explicit
in each kernel launch. Therefore,
interpret the SDPA line as the framework's practical automatic baseline, not as proof of one fixed backend or
one fixed internal arithmetic path.

In [ ]:
BENCHMARK_PROVIDERS = ["triton_v1", "triton_v2", "sdpa_auto"]
PROVIDER_LABELS = ["Triton V1", "Triton V2", "PyTorch SDPA (auto)"]
PROVIDER_STYLES = [("blue", "-"), ("green", "-"), ("red", "--")]


def make_benchmark(dtype, input_precision, title, plot_name):
    @triton.testing.perf_report(
        triton.testing.Benchmark(
            x_names=["seq_len"],
            x_vals=[2**i for i in range(9, 14)],
            line_arg="provider",
            line_vals=BENCHMARK_PROVIDERS,
            line_names=PROVIDER_LABELS,
            styles=PROVIDER_STYLES,
            ylabel="TFLOP/s",
            plot_name=plot_name,
            args={"dtype": dtype, "input_precision": input_precision},
        )
    )
    def benchmark(seq_len, provider, dtype, input_precision):
        B, H, D = 4, 8, 64
        # Give PyTorch SDPA the same FP32 arithmetic as the Triton kernels in this suite.
        torch.backends.cuda.matmul.fp32_precision = "tf32" if input_precision == "tf32" else "ieee"
        q, k, v = (
            torch.randn((B, H, seq_len, D), device=DEVICE, dtype=dtype)
            for _ in range(3)
        )

        if provider == "sdpa_auto":
            fn = lambda: F.scaled_dot_product_attention(q, k, v, is_causal=True)
        else:
            version = "v1_branch" if provider == "triton_v1" else "v2_staged"
            fn = lambda: flash_attention_fwd(
                q,
                k,
                v,
                version=version,
                input_precision=input_precision,
            )

        ms, ms_min, ms_max = triton.testing.do_bench(fn, quantiles=[0.5, 0.2, 0.8])
        causal_flops = 2.0 * B * H * seq_len * seq_len * D
        to_tflops = lambda elapsed_ms: causal_flops * 1e-12 / (elapsed_ms * 1e-3)
        return to_tflops(ms), to_tflops(ms_max), to_tflops(ms_min)

    benchmark.__name__ = plot_name
    benchmark.__doc__ = title
    return benchmark


BENCHMARK_SUITES = {
    "fp16": make_benchmark(
        torch.float16,
        "ieee",
        "FP16 storage (the FP32 dot precision switch does not apply)",
        "causal-attention-fp16",
    ),
    "fp32_ieee": make_benchmark(
        torch.float32,
        "ieee",
        "FP32 storage with strict IEEE Triton dot products",
        "causal-attention-fp32-ieee",
    ),
    "fp32_tf32": make_benchmark(
        torch.float32,
        "tf32",
        "FP32 storage with Triton TF32 dot products; PyTorch TF32 is allowed",
        "causal-attention-fp32-tf32",
    ),
}

print("PyTorch automatic SDPA is a dispatcher; its selected backend is not forced here.")
print("FP32 suites: Triton input_precision and torch fp32_precision are both 'ieee' or both 'tf32'.")

# Mark.run returns None by default, so return_df=True is required by the plotting cell.
benchmark_results = {
    mode: report.run(show_plots=False, print_data=True, return_df=True)
    for mode, report in BENCHMARK_SUITES.items()
}
torch.backends.cuda.matmul.fp32_precision = "ieee"  # restore the strict default

Configs picked by the autotuner (last tuned shape):

In [ ]:
for name, kern in KERNELS.items():
    cfg = kern.best_config
    print(f"{name:10s} {cfg.kwargs}  num_warps={cfg.num_warps}  num_stages={cfg.num_stages}")

In [ ]:
import os

STYLE = {
    "Triton V1": dict(color="#eb6834", marker="s"),
    "Triton V2": dict(color="#2a78d6", marker="o"),
    "PyTorch SDPA (auto)": dict(color="#1baf7a", marker="^", linestyle="--"),
}
BENCHMARK_TITLES = {
    "fp16": "FP16 storage",
    "fp32_ieee": "FP32 IEEE",
    "fp32_tf32": "FP32 TF32",
}


def benchmark_to_long(result):
    """Convert Triton's wide TFLOP/s table to rows with throughput and latency.

    Latency is derived, not separately measured: ms = FLOPs / (TFLOP/s * 1e9).
    """
    B, H, D = 4, 8, 64
    rows = []
    for _, result_row in result.iterrows():
        seq_len = int(result_row["seq_len"])
        causal_flops = 2.0 * B * H * seq_len * seq_len * D
        for implementation in PROVIDER_LABELS:
            tflops = float(result_row[f"{implementation} (TFLOP/s)"])
            latency_ms = causal_flops * 1e-9 / tflops
            rows.append({
                "seq_len": seq_len,
                "implementation": implementation,
                "latency_ms": latency_ms,
                "tflops": tflops,
            })
    return pd.DataFrame(rows)


def plot_benchmark(result, mode, gpu_name, output_dir="assets"):
    benchmark = benchmark_to_long(result)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), constrained_layout=True)
    panels = (
        (axes[0], "latency_ms", "Latency (ms, log scale) — lower is better", True),
        (axes[1], "tflops", "Throughput (TFLOP/s) — higher is better", False),
    )
    for axis, metric, title, log_y in panels:
        for implementation, style in STYLE.items():
            data = benchmark[benchmark.implementation == implementation].dropna()
            if data.empty:
                continue
            axis.plot(
                data.seq_len,
                data[metric],
                label=implementation,
                linewidth=2,
                markersize=7,
                **style,
            )
        axis.set_xscale("log", base=2)
        if log_y:
            axis.set_yscale("log")
        axis.set_xticks(sorted(benchmark.seq_len.unique()))
        axis.get_xaxis().set_major_formatter(plt.ScalarFormatter())
        axis.set_xlabel("sequence length")
        axis.set_title(title, fontsize=10, loc="left")
        axis.grid(True, which="major", color="#e6e6e6", linewidth=0.8)
        for side in ("top", "right"):
            axis.spines[side].set_visible(False)
    axes[0].legend(frameon=False, fontsize=9)
    fig.suptitle(
        f"Causal attention forward · {BENCHMARK_TITLES[mode]} · B=4, H=8, d=64 · {gpu_name}",
        fontsize=11,
        x=0.01,
        ha="left",
    )

    os.makedirs(output_dir, exist_ok=True)
    stem = f"benchmark_{mode}_l40s"
    figure_path = os.path.join(output_dir, f"{stem}.png")
    csv_path = os.path.join(output_dir, f"{stem}.csv")
    fig.savefig(figure_path, dpi=150, bbox_inches="tight")
    benchmark.to_csv(csv_path, index=False)
    plt.show()
    return benchmark


benchmark_tables = {
    mode: plot_benchmark(result, mode, props.name)
    for mode, result in benchmark_results.items()
}

## 7. Discussion & next steps

### What separates V1 and V2

| | V1 (branch) | V2 (staged) |
|---|---|---|
| Causal check | runtime `if` on every KV tile | resolved at compile time (`STAGE: tl.constexpr`) |
| Off-diagonal tiles | masked loads + boundary `tl.where` | plain loads, no masking |
| Diagonal tile | causal `tl.where` + boundary `tl.where` | one combined `tl.where` |
| Code shape | one kernel body | kernel + reusable `_attn_inner` helper |

Since only $2/(n+1)$ of the computed tiles sit on the diagonal (Section 2), V2 removes masking work from almost
every tile and gives the compiler a cleaner loop body. Whether that produces a measurable speedup depends on
shape, dtype, selected autotune configuration, and GPU.

### Precision on the L40S (`sm_89`)

- TF32 is an arithmetic mode, not a tensor storage dtype: Q, K, and V remain `torch.float32`.
- Triton receives `input_precision="tf32"` or `"ieee"` explicitly for both $QK^\top$ and $PV$.
- TF32 uses tensor cores and trades mantissa precision for throughput; the correctness table therefore uses a
  looser tolerance than strict FP32 IEEE.
- The selector does not change FP16 behavior because FP16 already uses its own tensor-core path.
- PyTorch's `fp32_precision` is set per suite so SDPA uses the same FP32 arithmetic as Triton.
- PyTorch SDPA remains an automatic dispatcher, so its chosen backend may differ across versions and GPUs.

### Profiling with Nsight Compute

Matmul FLOPs ignore the softmax, but its `exp2` runs on the SFU and can cost 25–50% of the matmul time.
To see which unit limits a kernel, profile one launch and compare pipe utilisation:

```bash
ncu --kernel-name regex:_fwd_v2 --launch-skip 5 --launch-count 1 --metrics \
  sm__pipe_tensor_op_hmma_cycles_active.avg.pct_of_peak_sustained_active,\
  sm__inst_executed_pipe_xu.avg.pct_of_peak_sustained_active,\
  sm__inst_executed_pipe_fma.avg.pct_of_peak_sustained_active,\
  sm__inst_executed_pipe_alu.avg.pct_of_peak_sustained_active \
  python bench.py
```

- **Tensor pipe high, XU moderate:** matmul-bound, which is the healthy case.
- **XU near peak, tensor pipe idle:** `exp2`-bound; the next step is overlapping softmax with the GEMMs
  (FA3-style warp specialisation / ping-pong).
- **Neither busy:** memory- or latency-bound; check `cp.async` waits, shared-memory stalls and occupancy.

### Possible next steps
- [ ] **Backward pass** — `L` is already returned, which is what the FlashAttention backward needs.
- [x] Fold $\log_2 e$ into the softmax scale to keep the hot softmax loop fully in base 2 (one FMA + one `exp2`).
- [ ] Profile V1/V2 with Nsight Compute (above) and report utilisation against the L40S dense tensor-core peak.
- [ ] Non-causal mode and variable-length (packed) sequences.
- [ ] GQA / MQA (fewer KV heads than Q heads).
- [ ] Wrap in `torch.autograd.Function` once the backward kernel exists.

### References
1. T. Dao et al., *FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness*, NeurIPS 2022. [arXiv:2205.14135](https://arxiv.org/abs/2205.14135)
2. T. Dao, *FlashAttention-2: Faster Attention with Better Parallelism and Work Partitioning*, 2023. [arXiv:2307.08691](https://arxiv.org/abs/2307.08691)
3. M. Milakov, N. Gimelshein, *Online normalizer calculation for softmax*, 2018. [arXiv:1805.02867](https://arxiv.org/abs/1805.02867)
4. Triton tutorial: [Fused Attention](https://triton-lang.org/main/getting-started/tutorials/06-fused-attention.html)